# Agent Debugging

이 노트북은 기존 workflow 위에 debugging 튜토리얼을 추가한다. 원래 노트북이나 main workflow architecture를 바꾸지 않고, trace log, execution inspection, failure debugging을 더 잘 보는 렌즈를 제공하는 것이 목적이다.

## 학습 목표

- agent debugging에서 trace가 왜 중요한지 이해한다.
- node 단위 입력과 출력을 점검하는 방법을 익힌다.
- 정상 answered run과 약한 run 또는 abstained run을 비교한다.
- failure evidence를 다음 개선 아이디어로 연결한다.


## 개념 설명

trace inspection과 artifact 생성은 예상한 `uv` 환경 안에서 이루어져야 의미가 있다. 재현 가능한 디버깅(reproducible debugging)은 결국 재현 가능한 interpreter에서 시작한다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

이 setup 셀은 `src/trace_debug.py`의 additive helper를 가져오고, 기존 workflow와 evaluation 코드를 그대로 재사용한다. workflow 자체를 뜯어고치는 것이 아니라, 더 좋은 debugging view를 덧씌우는 방식이다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.evaluator import extract_failure_cases, run_evaluation_suite
from src.ingestion import build_demo_index
from src.trace_debug import display_node_inputs, display_node_outputs, display_trace
from src.workflow import run_workflow

pd.set_option('display.max_colwidth', 140)
retriever = build_demo_index(persist=False)


trace log가 중요한 이유는 agent behavior가 여러 단계로 쪼개져 있기 때문이다. 최종 답만 보고는 retrieval, planning, tool use, verification, fallback 중 어디서 문제가 생겼는지 알기 어렵다. 좋은 trace는 각 단계를 inspectable하게 만든다.

이 notebook은 세 가지 레벨에 집중한다.

- 전체 trace overview
- 특정 node의 입력
- 특정 node의 출력

## 구현


In [ ]:
happy_state = run_workflow('How many days are in the pilot window?', retriever=retriever)
abstain_state = run_workflow('Who is the current CEO of the company?', retriever=retriever)
{
    'happy_status': happy_state['final_status'],
    'abstain_status': abstain_state['final_status'],
    'happy_steps': len(happy_state['trace']),
    'abstain_steps': len(abstain_state['trace']),
}


첫 번째 debugging view는 전체 trace 테이블이다. 이 버전은 node 이름, 직렬화된 입력, 직렬화된 출력, timestamp, 그리고 timestamp 기반 latency까지 함께 보여준다.


In [ ]:
debug_trace_frame = display_trace(happy_state['trace'], render=False)
debug_trace_frame


## Trace 성능 분석(Trace Performance Analysis)

trace debugging은 correctness만 보는 작업이 아니다. node별 latency를 보면 workflow가 커질수록 retrieval, tool execution, synthesis, verification 중 어디가 bottleneck이 되는지 읽을 수 있다. 아래 표는 node, 측정된 latency, 그리고 직렬화된 inputs/outputs에 집중한다.


In [ ]:
performance_frame = display_trace(happy_state['trace'], render=False)[['node', 'latency', 'inputs', 'outputs']]
performance_frame

latency가 높다고 해서 자동으로 나쁜 것은 아니다. 중요한 질문은 그 시간이 해당 작업에 비해 자연스러운지 여부다. 예를 들어 retrieval이 normalization보다 느린 것은 자연스럽지만, tool이나 synthesis가 비정상적으로 느리다면 디버깅 포인트가 된다.


이제 어떤 node가 중요해 보이는지 알았다면, 더 좁게 보는 것이 좋다. 다음 두 셀은 선택한 node의 입력만, 그리고 출력만 따로 점검하게 해준다.


In [ ]:
display_node_inputs(happy_state['trace'], 'retrieve_docs', render=False)


node 출력은 입력만큼 중요하다. 기대했던 동작이 아니라, workflow가 실제로 무엇을 만들어냈는지를 확인할 수 있기 때문이다.


In [ ]:
display_node_outputs(happy_state['trace'], 'retrieve_docs', render=False)


## 실험

좋은 debugging 실험은 grounded하게 answered 된 run과 abstained run을 나란히 비교하는 것이다. 두 trace는 생각보다 많이 다르지 않을 수 있고, 실제 차이는 classification, evidence coverage, fallback behavior에 숨어 있는 경우가 많다.


In [ ]:
comparison = pd.DataFrame(
    [
        {
            'query': happy_state['user_query'],
            'final_status': happy_state['final_status'],
            'coverage_score': happy_state['verification_result'].coverage_score,
            'unsupported_claims': len(happy_state['verification_result'].unsupported_claims),
        },
        {
            'query': abstain_state['user_query'],
            'final_status': abstain_state['final_status'],
            'coverage_score': abstain_state['verification_result'].coverage_score,
            'unsupported_claims': len(abstain_state['verification_result'].unsupported_claims),
        },
    ]
)
comparison


디버깅은 반복 evaluation과 함께 볼 때 더 강해진다. 이 셀은 작은 evaluation pass를 돌리고 failure row만 뽑아서, 개별 trace가 아니라 dataset 수준에서 무엇이 반복적으로 문제인지 보여준다.


In [ ]:
results, summary = run_evaluation_suite(repeats=1, persist_outputs=True)
failures = extract_failure_cases(results)
failures[['system', 'question_id', 'question', 'failure_type', 'predicted_status']].head(10)


## 결과 해석

이 debugging view가 보여주는 핵심은 두 가지다. trace는 한 번의 실행을 깊게 설명해주고, evaluation failure는 전체적인 패턴을 설명해준다. 둘을 같이 봐야 debugging이 체계적(systematic)인 작업이 된다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'debugging_view': 'full_trace', 'use_case': 'understand the whole execution path'},
        {'debugging_view': 'node_inputs', 'use_case': 'inspect what information reached a node'},
        {'debugging_view': 'node_outputs', 'use_case': 'inspect what the node actually produced'},
        {'debugging_view': 'failure_table', 'use_case': 'spot recurring issues across many runs'},
    ]
)
analysis_frame


## 핵심 정리

- 이 실험을 통해 trace는 숨겨진 agent behavior를 inspectable한 근거로 바꿔준다는 점을 확인했다.
- 특정 stage가 의심될 때는 node 단위 inspection이 특히 유용하다.
- evaluation failure는 trace를 보완하면서 반복적인 약점을 보여준다.
- 면접에서는 debugging 경험을 설명할 때 **trace, node inspection, repeated failure pattern**을 함께 말하면 시스템적으로 문제를 다뤘다는 인상을 줄 수 있다.
